<a href="https://colab.research.google.com/github/maham-bhatti/Traffic-Control-System/blob/main/MAPPO_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ══════════════════════════════════════════════════════════════
# SETUP CELL — GitHub → MAPPO (flat zip, files directly inside)
# ══════════════════════════════════════════════════════════════
import subprocess, shutil, os, zipfile, sys

# Step 1 — Clone repo
if os.path.exists('/content/repo'):
    shutil.rmtree('/content/repo')

print('Cloning from GitHub...')
subprocess.run([
    'git', 'clone',
    'https://github.com/maham-bhatti/Traffic-Control-System.git',
    '/content/repo'
], check=True)
print('✓ Repo cloned')

# Step 2 — Extract flat zip directly to /content
zip_path = '/content/repo/MAPPO.zip'
print('\nExtracting MAPPO.zip...')
with zipfile.ZipFile(zip_path, 'r') as z:
    print('── Files inside zip ──')
    for name in sorted(z.namelist()):
        print(f'  {name}')
    z.extractall('/content/')
print('✓ Extracted directly to /content')

# Step 3 — Python path
sys.path.insert(0, '/content')

# Step 4 — Verify
REQUIRED = [
    'gcn_encoder.py', 'mappo_atsc.py', 'traci_env_new.py',
    'manhattan.net.xml', 'run.sumocfg',
    'final_routes.rou.xml', 'low_routes.rou.xml',
    'medium_routes.rou.xml', 'high_routes.rou.xml',
]
print('\n── Required files check ──')
all_ok = True
for f in REQUIRED:
    exists = os.path.exists(f'/content/{f}')
    size   = f'{os.path.getsize(f"/content/{f}")/1024:.1f} KB' if exists else ''
    if not exists: all_ok = False
    print(f'  {"✓" if exists else "✗ MISSING":<12} {f:<35} {size}')

print()
print('✅ All set! Now run Cell 1.' if all_ok else '❌ Some files missing — check zip contents above!')

# GCN-MAPPO Adaptive Traffic Signal Control
## Manhattan 4×4 Grid — Kaggle Training
**Run cells one by one in order. Do not skip.**

Dataset path: `/content/dataset'

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Install SUMO (Internet must be ON, takes ~2-3 min)
# ══════════════════════════════════════════════════════════════
import subprocess, sys, os

print('Adding SUMO stable PPA...')
subprocess.run(['add-apt-repository', 'ppa:sumo/stable', '-y'],
               capture_output=True, text=True)

print('Updating apt...')
subprocess.run(['apt-get', 'update', '-q'],
               capture_output=True, text=True)

print('Installing SUMO...')
r = subprocess.run(
    ['apt-get', 'install', '-y', 'sumo', 'sumo-tools', 'sumo-doc'],
    capture_output=True, text=True
)
print(r.stdout[-500:] if r.stdout else '')
if r.returncode != 0:
    print('apt stderr:', r.stderr[-300:])

# Set SUMO_HOME BEFORE importing traci anywhere
os.environ['SUMO_HOME'] = '/usr/share/sumo'
if '/usr/share/sumo/tools' not in sys.path:
    sys.path.insert(0, '/usr/share/sumo/tools')

# Kill any stale SUMO processes
subprocess.run(['pkill', '-9', '-f', 'sumo'],  capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'traci'], capture_output=True)

# Force clear any cached traci modules
for mod in list(sys.modules.keys()):
    if 'traci' in mod or 'sumo' in mod:
        del sys.modules[mod]

# Verify installation
ver = subprocess.run(['sumo', '--version'], capture_output=True, text=True)
if 'SUMO' in ver.stdout:
    print('\n✓ SUMO installed:', ver.stdout.split('\n')[0])
else:
    print('✗ ERROR: SUMO not found. Check stderr below:')
    print(ver.stderr)


In [ ]:
# CELL 2: Files already in /content from setup cell — just verify
import os

WORK_DIR = '/content'
os.chdir(WORK_DIR)

print('Files already loaded from GitHub zip.')
print('\nFiles in /content:')
for f in sorted(os.listdir(WORK_DIR)):
    if not f.startswith('.') and os.path.isfile(os.path.join(WORK_DIR, f)):
        size = f'{os.path.getsize(os.path.join(WORK_DIR, f))/1024:.1f} KB'
        print(f'  {f:<40} {size}')

print('\n✓ Working directory ready — proceed to Cell 3')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3 — Verify traci_env_new.py and fix run.sumocfg paths
# ══════════════════════════════════════════════════════════════
import os

WORK_DIR = '/content'

# Check traci_env_new.py version
src = open(os.path.join(WORK_DIR, 'traci_env_new.py')).read()
checks = [
    ('_safe_close',            'has _safe_close helper'),
    ('getControlledLanes',     'uses correct TraCI lane API'),
    ('DENSITY_ROUTE_FILES',    'has multi-density support'),
    ('_resolve_config',        'has density config resolver'),
]
print('traci_env_new.py checks:')
for token, label in checks:
    found = token in src
    print(f'  {"✓" if found else "~"} {label}')

# Fix run.sumocfg — make sure it uses relative paths (working dir)
cfg_path = os.path.join(WORK_DIR, 'run.sumocfg')
with open(cfg_path) as f:
    cfg_content = f.read()
print(f'\nrun.sumocfg contents:')
print(cfg_content)

# Patch if cfg points to old dataset path
if '/content' in cfg_content:
    import re
    # Strip any absolute path, keep just the filename
    cfg_content = re.sub(r'[^"]+/([^/"]+\.(?:xml|rou\.xml))', r'\1', cfg_content)
    with open(cfg_path, 'w') as f:
        f.write(cfg_content)
    print('\n✓ Patched run.sumocfg — removed absolute dataset paths')
    print(cfg_content)
else:
    print('\n✓ run.sumocfg looks fine (no absolute dataset paths)')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4 — Install Python packages
# ══════════════════════════════════════════════════════════════
import subprocess, sys

pkgs = ['torch', 'numpy', 'pandas', 'matplotlib']
print(f'Installing: {pkgs} ...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install'] + pkgs + ['-q'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('pip stderr:', r.stderr[-500:])
else:
    print('✓ Python packages ready')

# Quick import check
import importlib
for pkg in pkgs:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  ✓ {pkg} {ver}')
    except ImportError as e:
        print(f'  ✗ {pkg} — {e}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — Quick environment test  (50 steps, not real training)
# ══════════════════════════════════════════════════════════════
import os, sys, numpy as np, subprocess, time

# ── 1. Paths (must be set BEFORE any traci/gcn import) ────────
WORK_DIR = '/content'
os.environ['SUMO_HOME'] = '/usr/share/sumo'
for p in [WORK_DIR, '/usr/share/sumo/tools']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(WORK_DIR)

# ── 2. Kill any stale SUMO process ────────────────────────────
subprocess.run(['pkill', '-f', 'sumo'], capture_output=True)
time.sleep(1.0)

# ── 3. Import project modules ─────────────────────────────────
from gcn_encoder import REAL_JUNCTIONS, RAW_OBS_DIM, ACT_DIM
from traci_env_new   import SUMOEnv, SUMO_CFG
#_SCRIPTS_DIR = '/usr/share/sumo/tools'

print(f'SUMO config    : {SUMO_CFG}')
print(f'Config exists  : {os.path.exists(SUMO_CFG)}')
print(f'Working dir    : {WORK_DIR}')
print()

# ── 4. Run short test (warmup=50, ep=100 so it's fast) ────────
print('Running 50-step environment test ...')
env = SUMOEnv(warmup=50, ep_duration=100)
try:
    obs = env.reset()
    rewards = []
    for _ in range(50):
        acts = {jid: np.random.randint(0, ACT_DIM[jid]) for jid in REAL_JUNCTIONS}
        obs, rews, done, info = env.step(acts)
        rewards.extend(rews.values())
        if done:
            break
    print(f'Reward range: min={min(rewards):.3f}  max={max(rewards):.3f}  mean={np.mean(rewards):.3f}')
    print('✓ Environment test PASSED')
except Exception as e:
    print(f'✗ Environment test FAILED: {e}')
    raise
finally:
    env.close()
    subprocess.run(['pkill', '-f', 'sumo'], capture_output=True)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — Stage 0: Baseline training  (final_routes.rou.xml)
# Set EPISODES= 500 for real training
# ══════════════════════════════════════════════════════════════
import os, sys, torch, subprocess, time

WORK_DIR = '/content'
os.environ['SUMO_HOME'] = '/usr/share/sumo'
for p in [WORK_DIR, '/usr/share/sumo/tools']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(WORK_DIR)

# Kill stale SUMO before starting
subprocess.run(['pkill', '-f', 'sumo'], capture_output=True)
time.sleep(1.5)

from traci_env_new import train

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 500
print(f'Device: {DEVICE}  |  Episodes: {EPISODES}')

ctrl = train(
    n_episodes = EPISODES,
    device     = DEVICE,
    save_dir   = '/content/ckpt_base',
    use_gui    = False,
    curriculum = False,
    density    = None,     # uses final_routes.rou.xml (DEFAULT_ROUTE_FILE)
)

In [ ]:
# Download Base Density Results
import os, shutil, zipfile
from google.colab import files as colab_files

ckpt_dir = '/content/ckpt_base'
log_path = f'{ckpt_dir}/training_log.csv'

# Create ZIP with Base density files
zip_path = '/content/base_density_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Training log
    if os.path.exists(log_path):
        zf.write(log_path, 'training_log.csv')
        print('Added: training_log.csv')

    # All checkpoint files
    if os.path.exists(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            full_path = os.path.join(ckpt_dir, f)
            zf.write(full_path, f'checkpoint/{f}')
            print(f'Added: {f}')

print(f'\nZIP saved  : {zip_path}')
print(f'ZIP size   : {os.path.getsize(zip_path)/1024:.1f} KB')
display(FileLink('base_density_results.zip',
                 result_html_prefix='Download: '))

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7 — Stage 1: Off-peak training  (low density)
# Resumes from Stage 0 checkpoint
# Set EPISODES=700 for real training
# ══════════════════════════════════════════════════════════════
import os, sys, torch, subprocess, time

WORK_DIR = '/content'
os.environ['SUMO_HOME'] = '/usr/share/sumo'
for p in [WORK_DIR, '/usr/share/sumo/tools']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(WORK_DIR)

# Kill stale SUMO before starting
subprocess.run(['pkill', '-f', 'sumo'], capture_output=True)
time.sleep(1.5)

from traci_env_new import train

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 700

# Check that low_routes.rou.xml exists; if not, fall back to default
if os.path.exists('/content/low_routes.rou.xml'):
    density = 'low'
    print(f'✓ low_routes.rou.xml found — using low density')
else:
    density = None
    print('⚠ low_routes.rou.xml not found — falling back to default routes')

print(f'Device: {DEVICE}  |  Episodes: {EPISODES}  |  Density: {density or "default"}')

# Check resume checkpoint
resume = '/content/ckpt_base'
if not os.path.isdir(resume):
    print(f'⚠ Checkpoint not found at {resume} — starting fresh')
    resume = None
else:
    print(f'✓ Resuming from {resume}')

ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/content/ckpt_low',
    resume_from = resume,
    use_gui     = False,
    curriculum  = False,
    density     = density,
)

In [ ]:
# Download Low Density Results
import os, shutil, zipfile
from google.colab import files as colab_files

ckpt_dir = '/content/ckpt_low'
log_path = f'{ckpt_dir}/training_log.csv'

# Create ZIP with Low density files
zip_path = '/content/low_density_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Training log
    if os.path.exists(log_path):
        zf.write(log_path, 'training_log.csv')
        print('Added: training_log.csv')

    # All checkpoint files
    if os.path.exists(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            full_path = os.path.join(ckpt_dir, f)
            zf.write(full_path, f'checkpoint/{f}')
            print(f'Added: {f}')

print(f'\nZIP saved  : {zip_path}')
print(f'ZIP size   : {os.path.getsize(zip_path)/1024:.1f} KB')
display(FileLink('low_density_results.zip',
                 result_html_prefix='Download: '))

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8 — Stage 2: Moderate traffic training
# Resumes from Stage 1 checkpoint
# Set EPISODES=800 for real training
# ══════════════════════════════════════════════════════════════
import os, sys, torch, subprocess, time

WORK_DIR = '/content'
os.environ['SUMO_HOME'] = '/usr/share/sumo'
for p in [WORK_DIR, '/usr/share/sumo/tools']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(WORK_DIR)

subprocess.run(['pkill', '-f', 'sumo'], capture_output=True)
time.sleep(1.5)

from traci_env_new import train

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 800

if os.path.exists('/content/medium_routes.rou.xml'):
    density = 'medium'
    print('✓ medium_routes.rou.xml found')
else:
    density = None
    print('⚠ medium_routes.rou.xml not found — falling back to default routes')

resume = '/content/weights/low/ckpt_low_best_download'
if not os.path.isdir(resume):
    print(f'⚠ Checkpoint not found at {resume} — starting fresh')
    resume = None
else:
    print(f'✓ Resuming from {resume}')

print(f'Device: {DEVICE}  |  Episodes: {EPISODES}  |  Density: {density or "default"}')

ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/content/ckpt_med',
    resume_from = resume,
    use_gui     = False,
    curriculum  = False,
    density     = 'medium',
)

In [ ]:
# Download Medium Density Results
import os, shutil, zipfile
from google.colab import files as colab_files

ckpt_dir = '/content/ckpt_med'
log_path = f'{ckpt_dir}/training_log.csv'

# Create ZIP with Medium density files
zip_path = '/content/med_density_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Training log
    if os.path.exists(log_path):
        zf.write(log_path, 'training_log.csv')
        print('Added: training_log.csv')

    # All checkpoint files
    if os.path.exists(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            full_path = os.path.join(ckpt_dir, f)
            zf.write(full_path, f'checkpoint/{f}')
            print(f'Added: {f}')

print(f'\nZIP saved  : {zip_path}')
print(f'ZIP size   : {os.path.getsize(zip_path)/1024:.1f} KB')
display(FileLink('med_density_results.zip',
                 result_html_prefix='Download: '))

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9 — Stage 3: Peak-hour training
# Resumes from Stage 2 checkpoint
# Set EPISODES=1000 for real training
# ══════════════════════════════════════════════════════════════
import os, sys, torch, subprocess, time

WORK_DIR = '/content'
os.environ['SUMO_HOME'] = '/usr/share/sumo'
for p in [WORK_DIR, '/usr/share/sumo/tools']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(WORK_DIR)

subprocess.run(['pkill', '-f', 'sumo'], capture_output=True)
time.sleep(1.5)

from traci_env_new import train

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 1000

if os.path.exists('/content/high_routes.rou.xml'):
    density = 'high'
    print('✓ high_routes.rou.xml found')
else:
    density = None
    print('⚠ high_routes.rou.xml not found — falling back to default routes')

resume = '/content/weights/medium/ckpt_med_best_download'
if not os.path.isdir(resume):
    print(f'⚠ Checkpoint not found at {resume} — starting fresh')
    resume = None
else:
    print(f'✓ Resuming from {resume}')

print(f'Device: {DEVICE}  |  Episodes: {EPISODES}  |  Density: {density or "default"}')

ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/content/ckpt_high',
    resume_from = resume,
    use_gui     = False,
    curriculum  = False,
    density     = 'high',
)

In [ ]:
# Download High Density Results
import os, shutil, zipfile
from google.colab import files as colab_files

ckpt_dir = '/content/ckpt_high'
log_path = f'{ckpt_dir}/training_log.csv'

# Create ZIP with High density files
zip_path = '/content/high_density_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Training log
    if os.path.exists(log_path):
        zf.write(log_path, 'training_log.csv')
        print('Added: training_log.csv')

    # All checkpoint files
    if os.path.exists(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            full_path = os.path.join(ckpt_dir, f)
            zf.write(full_path, f'checkpoint/{f}')
            print(f'Added: {f}')

print(f'\nZIP saved  : {zip_path}')
print(f'ZIP size   : {os.path.getsize(zip_path)/1024:.1f} KB')
display(FileLink('high_density_results.zip',
                 result_html_prefix='Download: '))

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10 — Final evaluation across all 3 densities
# ══════════════════════════════════════════════════════════════
import os, sys, torch, subprocess, time

WORK_DIR = '/content'
os.environ['SUMO_HOME'] = '/usr/share/sumo'
for p in [WORK_DIR, '/usr/share/sumo/tools']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(WORK_DIR)

subprocess.run(['pkill', '-f', 'sumo'], capture_output=True)
time.sleep(1.5)

from traci_env_new import _evaluate
from mappo_atsc import MAPPOController

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Pick best available checkpoint
for ckpt in ['/content/ckpt_high_best',
             '/content/ckpt_high_final',
             '/content/ckpt_high',
             '/content/ckpt_med',
             '/content/ckpt_base']:
    if os.path.isdir(ckpt):
        CKPT_DIR = ckpt
        break
else:
    raise FileNotFoundError('No checkpoint found. Run training cells first.')

print(f'Loading from: {CKPT_DIR}')
ctrl = MAPPOController(device=DEVICE)
ctrl.load(CKPT_DIR)

print('\n=== Evaluation Results (same model on all densities) ===')
for density, label in [
    ('low',    'Off-peak  (~700 veh/hr)'),
    ('Medium',     'Moderate  (~800 veh/hr)'),
    ('high',   'Peak-hour (~1000 veh/hr)'),
]:
    print(f'\nDensity: {label}')
    _evaluate(ctrl, n_reps=3, use_gui=False, density=density)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11 — Plot training curves for all stages
# ══════════════════════════════════════════════════════════════
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # headless backend — required on Kaggle
import matplotlib.pyplot as plt
import os

stages = [
    ('ckpt_base', 'Stage 0: Baseline'),
    ('ckpt_low',  'Stage 1: Off-peak'),
    ('ckpt_med',  'Stage 2: Moderate'),
    ('ckpt_high', 'Stage 3: Peak hour'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (ckpt, title) in zip(axes.flat, stages):
    log_path = f'/content/{ckpt}/training_log.csv'
    if os.path.exists(log_path):
        df = pd.read_csv(log_path)
        ax.plot(df['episode'], df['avg_reward'],      label='Reward',      color='steelblue')
        ax2 = ax.twinx()
        ax2.plot(df['episode'], df['avg_travel_time'], label='Travel time', color='darkorange')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Avg Reward', color='steelblue')
        ax2.set_ylabel('Avg Travel Time (s)', color='darkorange')
        ax.tick_params(axis='y', labelcolor='steelblue')
        ax2.tick_params(axis='y', labelcolor='darkorange')
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
    else:
        ax.text(0.5, 0.5, f'No data yet\n({ckpt})',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=12, color='gray')
        ax.set_title(title)

plt.suptitle('GCN-MAPPO Training Curves — All Stages', fontsize=14, fontweight='bold')
plt.tight_layout()
out = '/content/training_curves_all_stages.png'
plt.savefig(out, dpi=150)
plt.show()
print(f'✓ Plot saved: {out}')

In [ ]:
# Different plots for RP (Project)

"""
visualize_results.py
====================
Generates publication-quality figures for your thesis/paper.

Figures produced:
  Fig 1 — Travel time comparison bar chart (3 densities × 3 controllers)
  Fig 2 — Reward comparison bar chart
  Fig 3 — Queue length comparison
  Fig 4 — Training curves (reward + travel time per stage)
  Fig 5 — GCN-MAPPO improvement % over Fixed-Time (heatmap style)
  Fig 6 — Box plots — travel time distribution per controller
  Fig 7 — Density scalability line chart (how each controller scales with traffic)
  Fig 8 — Combined dashboard (all metrics in one figure — for paper appendix)

Usage (Kaggle notebook cell):
    exec(open('visualize_results.py').read())

Requires:
    /content/comparison_results.csv   (from baseline_comparison.py)
    /content/comparison_summary.csv   (from baseline_comparison.py)
    /content/ckpt_*/training_log.csv  (from training cells 6-9)

All figures saved to: /content/figures/
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── Style — clean academic look ───────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'legend.framealpha': 0.85,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
})

# ── Colors ────────────────────────────────────────────────────────────────────
C = {
    'Fixed-Time': '#E07B54',   # orange-red
    'Random':     '#A8A8A8',   # grey
    'GCN-MAPPO':  '#4C8EDA',   # blue
}
DENSITY_LABELS = {'low': 'Low\n(~824 veh/hr)', 'medium': 'Medium\n(~1685 veh/hr)', 'high': 'High\n(~3376 veh/hr)'}
DENSITY_ORDER  = ['low', 'medium', 'high']
CTRL_ORDER     = ['Fixed-Time', 'Random', 'GCN-MAPPO']

# ── Output dir ────────────────────────────────────────────────────────────────
FIG_DIR = '/content/figures'
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path)
    plt.close()
    print(f"  Saved → {path}")

# ── Load data ─────────────────────────────────────────────────────────────────
RESULTS_PATH = '/content/comparison_results.csv'
SUMMARY_PATH = '/content/comparison_summary.csv'

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        "Run baseline_comparison.py first to generate comparison_results.csv"
    )

df      = pd.read_csv(RESULTS_PATH)
summary = pd.read_csv(SUMMARY_PATH)


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 1 — Travel Time Comparison Bar Chart
# ══════════════════════════════════════════════════════════════════════════════
def fig_travel_time_bar():
    fig, ax = plt.subplots(figsize=(9, 5))

    n_density = len(DENSITY_ORDER)
    n_ctrl    = len(CTRL_ORDER)
    x         = np.arange(n_density)
    width     = 0.25
    offsets   = [-width, 0, width]

    for i, ctrl in enumerate(CTRL_ORDER):
        means, stds = [], []
        for d in DENSITY_ORDER:
            sub = summary[(summary['controller'] == ctrl) & (summary['density'] == d)]
            means.append(sub['mean_travel_time'].values[0] if len(sub) else 0)
            stds.append(sub['std_travel_time'].values[0]   if len(sub) else 0)
        bars = ax.bar(x + offsets[i], means, width, yerr=stds,
                      color=C[ctrl], label=ctrl, capsize=4, alpha=0.88,
                      error_kw={'elinewidth': 1.5})

    ax.set_xlabel('Traffic Density')
    ax.set_ylabel('Average Travel Time (seconds)')
    ax.set_title('Fig 1 — Average Travel Time by Controller and Traffic Density')
    ax.set_xticks(x)
    ax.set_xticklabels([DENSITY_LABELS[d] for d in DENSITY_ORDER])
    ax.legend(title='Controller')
    ax.set_ylim(bottom=0)

    savefig('fig1_travel_time_comparison.png')


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 2 — Reward Comparison Bar Chart
# ══════════════════════════════════════════════════════════════════════════════
def fig_reward_bar():
    fig, ax = plt.subplots(figsize=(9, 5))

    x      = np.arange(len(DENSITY_ORDER))
    width  = 0.25
    offsets = [-width, 0, width]

    for i, ctrl in enumerate(CTRL_ORDER):
        means, stds = [], []
        for d in DENSITY_ORDER:
            sub = summary[(summary['controller'] == ctrl) & (summary['density'] == d)]
            means.append(sub['mean_reward'].values[0] if len(sub) else 0)
            stds.append(sub['std_reward'].values[0]   if len(sub) else 0)
        ax.bar(x + offsets[i], means, width, yerr=stds,
               color=C[ctrl], label=ctrl, capsize=4, alpha=0.88,
               error_kw={'elinewidth': 1.5})

    ax.set_xlabel('Traffic Density')
    ax.set_ylabel('Average Episode Reward')
    ax.set_title('Fig 2 — Average Reward by Controller and Traffic Density')
    ax.set_xticks(x)
    ax.set_xticklabels([DENSITY_LABELS[d] for d in DENSITY_ORDER])
    ax.legend(title='Controller')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

    savefig('fig2_reward_comparison.png')


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 3 — Queue Length Comparison
# ══════════════════════════════════════════════════════════════════════════════
def fig_queue_bar():
    fig, ax = plt.subplots(figsize=(9, 5))

    x      = np.arange(len(DENSITY_ORDER))
    width  = 0.25
    offsets = [-width, 0, width]

    for i, ctrl in enumerate(CTRL_ORDER):
        means = []
        for d in DENSITY_ORDER:
            sub = summary[(summary['controller'] == ctrl) & (summary['density'] == d)]
            means.append(sub['mean_queue'].values[0] if len(sub) else 0)
        ax.bar(x + offsets[i], means, width,
               color=C[ctrl], label=ctrl, alpha=0.88)

    ax.set_xlabel('Traffic Density')
    ax.set_ylabel('Average Queue Length (vehicles)')
    ax.set_title('Fig 3 — Average Queue Length by Controller and Traffic Density')
    ax.set_xticks(x)
    ax.set_xticklabels([DENSITY_LABELS[d] for d in DENSITY_ORDER])
    ax.legend(title='Controller')
    ax.set_ylim(bottom=0)

    savefig('fig3_queue_comparison.png')


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 4 — Training Curves (all 4 stages)
# ══════════════════════════════════════════════════════════════════════════════
def fig_training_curves():
    stages = [
        ('ckpt_base',  'Stage 0: Baseline (default routes)'),
        ('ckpt_low',   'Stage 1: Off-peak (low density)'),
        ('ckpt_med',   'Stage 2: Moderate density'),
        ('ckpt_high',  'Stage 3: Peak hour (high density)'),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle('Fig 4 — GCN-MAPPO Training Curves Across All Stages', fontsize=14, y=1.01)

    for ax, (ckpt, title) in zip(axes.flat, stages):
        log_path = f'/content/{ckpt}/training_log.csv'
        if os.path.exists(log_path):
            log  = pd.read_csv(log_path)
            col  = 'avg_reward' if 'avg_reward' in log.columns else log.columns[1]
            col2 = 'avg_travel_time' if 'avg_travel_time' in log.columns else log.columns[2]

            ax2  = ax.twinx()
            ax.plot(log['episode'],  log[col],  color='#4C8EDA', label='Reward',       linewidth=1.8)
            ax2.plot(log['episode'], log[col2], color='#E07B54', label='Travel time', linewidth=1.8, linestyle='--')

            ax.set_title(title)
            ax.set_xlabel('Episode')
            ax.set_ylabel('Avg Reward', color='#4C8EDA')
            ax2.set_ylabel('Avg Travel Time (s)', color='#E07B54')
            ax.tick_params(axis='y', colors='#4C8EDA')
            ax2.tick_params(axis='y', colors='#E07B54')

            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)
        else:
            ax.text(0.5, 0.5, f'No training log found\n({ckpt})',
                    ha='center', va='center', transform=ax.transAxes, color='grey')
            ax.set_title(title)

    plt.tight_layout()
    savefig('fig4_training_curves.png')


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 5 — Improvement % of GCN-MAPPO over Fixed-Time
# ══════════════════════════════════════════════════════════════════════════════
def fig_improvement_heatmap():
    metrics = ['mean_travel_time', 'mean_queue', 'mean_wait']
    labels  = ['Travel Time', 'Queue Length', 'Wait Time']

    data = np.zeros((len(metrics), len(DENSITY_ORDER)))

    for di, density in enumerate(DENSITY_ORDER):
        fixed = summary[(summary['controller'] == 'Fixed-Time') & (summary['density'] == density)]
        mappo = summary[(summary['controller'] == 'GCN-MAPPO')  & (summary['density'] == density)]
        if len(fixed) and len(mappo):
            for mi, metric in enumerate(metrics):
                f_val = fixed[metric].values[0]
                m_val = mappo[metric].values[0]
                data[mi, di] = (f_val - m_val) / f_val * 100 if f_val != 0 else 0

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=-5, vmax=30)

    ax.set_xticks(range(len(DENSITY_ORDER)))
    ax.set_xticklabels(['Low', 'Medium', 'High'])
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)

    for i in range(len(labels)):
        for j in range(len(DENSITY_ORDER)):
            val = data[i, j]
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if abs(val) > 15 else 'black')

    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Improvement over Fixed-Time (%)')
    ax.set_title('Fig 5 — GCN-MAPPO Improvement over Fixed-Time Controller')
    ax.set_xlabel('Traffic Density')

    savefig('fig5_improvement_heatmap.png')


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 6 — Box Plots — Travel Time Distribution
# ══════════════════════════════════════════════════════════════════════════════
def fig_boxplots():
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
    fig.suptitle('Fig 6 — Travel Time Distribution per Controller', fontsize=13)

    for ax, density in zip(axes, DENSITY_ORDER):
        data_by_ctrl = []
        for ctrl in CTRL_ORDER:
            vals = df[(df['controller'] == ctrl) & (df['density'] == density)]['avg_travel_time'].values
            data_by_ctrl.append(vals)

        bp = ax.boxplot(data_by_ctrl, patch_artist=True, notch=False,
                        medianprops={'color': 'black', 'linewidth': 2})
        for patch, ctrl in zip(bp['boxes'], CTRL_ORDER):
            patch.set_facecolor(C[ctrl])
            patch.set_alpha(0.8)

        ax.set_title(f'{density.capitalize()} Density')
        ax.set_xticks(range(1, len(CTRL_ORDER) + 1))
        ax.set_xticklabels(CTRL_ORDER, rotation=15, ha='right')
        ax.set_ylabel('Travel Time (s)' if density == 'low' else '')
        ax.set_xlabel('Controller')

    plt.tight_layout()
    savefig('fig6_boxplots_travel_time.png')


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 7 — Scalability: How each controller handles increasing density
# ══════════════════════════════════════════════════════════════════════════════
def fig_scalability():
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Fig 7 — Controller Scalability with Traffic Density', fontsize=13)

    density_vals = [824, 1685, 3376]   # veh/hr

    for ctrl in CTRL_ORDER:
        tt_means, tt_stds = [], []
        rew_means = []
        for density in DENSITY_ORDER:
            sub = summary[(summary['controller'] == ctrl) & (summary['density'] == density)]
            tt_means.append(sub['mean_travel_time'].values[0] if len(sub) else 0)
            tt_stds.append(sub['std_travel_time'].values[0]   if len(sub) else 0)
            rew_means.append(sub['mean_reward'].values[0]     if len(sub) else 0)

        axes[0].errorbar(density_vals, tt_means, yerr=tt_stds,
                         label=ctrl, color=C[ctrl], marker='o',
                         linewidth=2, markersize=7, capsize=5)
        axes[1].plot(density_vals, rew_means,
                     label=ctrl, color=C[ctrl], marker='s',
                     linewidth=2, markersize=7)

    for ax, ylabel, title in zip(axes,
                                  ['Avg Travel Time (s)', 'Avg Reward'],
                                  ['Travel Time vs Traffic Volume', 'Reward vs Traffic Volume']):
        ax.set_xlabel('Traffic Volume (vehicles/hour)')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(density_vals)
        ax.set_xticklabels(['Low\n824', 'Medium\n1685', 'High\n3376'])
        ax.legend(title='Controller')

    plt.tight_layout()
    savefig('fig7_scalability.png')


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 8 — Combined Dashboard (for paper appendix)
# ══════════════════════════════════════════════════════════════════════════════
def fig_dashboard():
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle('GCN-MAPPO Adaptive Traffic Signal Control — Full Results Dashboard',
                 fontsize=15, fontweight='bold', y=1.01)

    gs = GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.38)

    # ── Row 1: per-density bar charts ─────────────────────────────────────────
    for col, density in enumerate(DENSITY_ORDER):
        ax = fig.add_subplot(gs[0, col])
        vals = [summary[(summary['controller']==c) & (summary['density']==density)]['mean_travel_time'].values[0]
                if len(summary[(summary['controller']==c) & (summary['density']==density)]) else 0
                for c in CTRL_ORDER]
        bars = ax.bar(CTRL_ORDER, vals, color=[C[c] for c in CTRL_ORDER], alpha=0.88)
        ax.set_title(f'{density.capitalize()} Density\nTravel Time')
        ax.set_ylabel('Seconds')
        ax.set_xticklabels(CTRL_ORDER, rotation=20, ha='right', fontsize=9)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{v:.0f}s', ha='center', va='bottom', fontsize=8)

    # ── Row 2: reward bars ────────────────────────────────────────────────────
    for col, density in enumerate(DENSITY_ORDER):
        ax = fig.add_subplot(gs[1, col])
        vals = [summary[(summary['controller']==c) & (summary['density']==density)]['mean_reward'].values[0]
                if len(summary[(summary['controller']==c) & (summary['density']==density)]) else 0
                for c in CTRL_ORDER]
        ax.bar(CTRL_ORDER, vals, color=[C[c] for c in CTRL_ORDER], alpha=0.88)
        ax.set_title(f'{density.capitalize()} Density\nReward')
        ax.set_ylabel('Avg Reward')
        ax.set_xticklabels(CTRL_ORDER, rotation=20, ha='right', fontsize=9)
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

    # ── Row 3: scalability lines ───────────────────────────────────────────────
    ax_tt  = fig.add_subplot(gs[2, 0:2])
    ax_imp = fig.add_subplot(gs[2, 2])

    density_vals = [824, 1685, 3376]
    for ctrl in CTRL_ORDER:
        tt_means = [summary[(summary['controller']==ctrl) & (summary['density']==d)]['mean_travel_time'].values[0]
                    if len(summary[(summary['controller']==ctrl) & (summary['density']==d)]) else 0
                    for d in DENSITY_ORDER]
        ax_tt.plot(density_vals, tt_means, label=ctrl, color=C[ctrl], marker='o', linewidth=2)

    ax_tt.set_xlabel('Traffic Volume (veh/hr)')
    ax_tt.set_ylabel('Avg Travel Time (s)')
    ax_tt.set_title('Scalability Across Densities')
    ax_tt.set_xticks(density_vals)
    ax_tt.legend(title='Controller', fontsize=9)

    # improvement bar
    imps = []
    for density in DENSITY_ORDER:
        sub = summary[(summary['controller']=='GCN-MAPPO') & (summary['density']==density)]
        imps.append(sub['improvement_vs_fixed'].values[0] if len(sub) and sub['improvement_vs_fixed'].values[0] else 0)

    bars = ax_imp.bar(['Low', 'Medium', 'High'], imps, color='#4C8EDA', alpha=0.88)
    ax_imp.set_title('GCN-MAPPO Improvement\nvs Fixed-Time (%)')
    ax_imp.set_ylabel('Improvement (%)')
    ax_imp.set_ylim(bottom=0)
    for bar, v in zip(bars, imps):
        ax_imp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    f'{v:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Legend patch
    patches = [mpatches.Patch(color=C[c], label=c, alpha=0.88) for c in CTRL_ORDER]
    fig.legend(handles=patches, loc='upper right', title='Controller',
               bbox_to_anchor=(1.02, 0.98), fontsize=10)

    savefig('fig8_dashboard.png')


# ══════════════════════════════════════════════════════════════════════════════
#  RUN ALL FIGURES
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("  Generating paper figures...")
print("=" * 60)

fig_travel_time_bar()
fig_reward_bar()
fig_queue_bar()
fig_training_curves()
fig_improvement_heatmap()
fig_boxplots()
fig_scalability()
fig_dashboard()

print("\n" + "=" * 60)
print(f"  All figures saved to: {FIG_DIR}")
print("=" * 60)
print("\n  Figure list:")
for f in sorted(os.listdir(FIG_DIR)):
    print(f"    {f}")

In [ ]:
# thesis results script. It compares 3 methods across 3 traffic densities
# Low          Medium        High
#Fixed-time:        320.5s        450.2s       680.1s   ← dumb baseline
#Random:            410.3s        520.1s       750.4s   ← random baseline
#GCN-MAPPO:         210.1s        290.5s       410.8s   ← your model
#────────────────────────────────────────────────────
#Improvement:        -34%          -35%         -40%    ← your contribution


import os, sys, numpy as np, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')

from traci_env_new import SUMOEnv
from gcn_encoder import REAL_JUNCTIONS, ACT_DIM
from mappo_atsc import MAPPOController

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_REPS = 3  # reps per method per density (increase to 5 for final results)

# ── Helper: run one episode and return travel time ────────────────
def run_episode(env, action_fn):
    obs = env.reset()
    for _ in range(2700):
        acts = action_fn(obs)
        obs, rews, done, info = env.step(acts)
        if done:
            break
    tt = env.avg_tt()
    env.close()
    return tt

# ── Method 1: Fixed-time (agent does nothing) ─────────────────────
def fixed_time_fn(obs):
    return {jid: 0 for jid in REAL_JUNCTIONS}

# ── Method 2: Random switching ────────────────────────────────────
def random_fn(obs):
    return {jid: np.random.randint(0, ACT_DIM[jid])
            for jid in REAL_JUNCTIONS}
# ── Method 3: GCN-MAPPO ──────────────────────────────────────────
def make_gcn_fn(ctrl):
    def gcn_fn(obs):
        aug = ctrl.gcn.augment_obs(obs)
        acts = {}
        for jid in REAL_JUNCTIONS:
            a, _, _ = ctrl.agents[jid].actor.get_action(
                torch.FloatTensor(aug[jid]).unsqueeze(0).to(DEVICE)
            )
            acts[jid] = int(a.item())
        return acts
    return gcn_fn

# ── Load best checkpoint ──────────────────────────────────────────
ckpt_dir = None
for name in ['ckpt_high_best', 'ckpt_high', 'ckpt_low_best', 'ckpt_base_best']:
    path = f'/content/{name}'
    if os.path.exists(path) and len([f for f in os.listdir(path) if f.endswith('.pt')]) == 14:
        ckpt_dir = path
        break

if ckpt_dir is None:
    print('ERROR: No valid checkpoint found in /content/')
    print('Upload your checkpoint to the dataset and re-run Cell 2')
else:
    print(f'Using checkpoint: {ckpt_dir}')
    ctrl = MAPPOController(device=DEVICE)
    ctrl.load(ckpt_dir)
    gcn_fn = make_gcn_fn(ctrl)

    # ── Run comparison across 3 densities ─────────────────────────
    densities = {
        'Low (824 veh/hr)':    '/content/run_low.sumocfg',
        'Medium (1685 veh/hr)': '/content/run_medium.sumocfg',
        'High (3376 veh/hr)':  '/content/run.sumocfg',
    }
# Write medium sumocfg if not exists
    if not os.path.exists('/content/run_medium.sumocfg'):
        with open('/content/run_medium.sumocfg', 'w') as f:
            f.write('<configuration><input>'
                    '<net-file value="/content/manhattan.net.xml"/>'
                    '<route-files value="/content/medium_routes.rou.xml"/>'
                    '</input><time><begin value="0"/></time></configuration>')

    if not os.path.exists('/content/run_low.sumocfg'):
        with open('/content/run_low.sumocfg', 'w') as f:
            f.write('<configuration><input>'
                    '<net-file value="/content/manhattan.net.xml"/>'
                    '<route-files value="/content/low_routes.rou.xml"/>'
                    '</input><time><begin value="0"/></time></configuration>')

    results = {}
    for density_label, cfg_path in densities.items():
        print(f'\n{"="*50}')
        print(f'Density: {density_label}')
        print(f'{"="*50}')
        results[density_label] = {}

        for method_name, action_fn in [
            ('Fixed-time', fixed_time_fn),
            ('Random',     random_fn),
            ('GCN-MAPPO',  gcn_fn),
        ]:
            times = []
            for rep in range(N_REPS):
                import traci_env_new
                traci_env_new.SUMO_CFG = cfg_path
                env = SUMOEnv(warmup=900, ep_duration=2700)
                tt = run_episode(env, action_fn)
                times.append(tt)
                print(f'  {method_name} rep {rep+1}: {tt:.1f}s')
            results[density_label][method_name] = {
                'mean': np.mean(times),
                'std':  np.std(times),
                'min':  np.min(times),
            }

    # ── Print final comparison table ──────────────────────────────
    print('\n' + '='*70)
    print('FINAL COMPARISON TABLE — Average Travel Time (seconds)')
    print('='*70)
    print(f'{"Method":<20}', end='')
    for d in densities:
        print(f'{d:<25}', end='')
    print()
    print('-'*70)

    for method in ['Fixed-time', 'Random', 'GCN-MAPPO']:
        print(f'{method:<20}', end='')
        for d in densities:
            m = results[d][method]['mean']
            s = results[d][method]['std']
            print(f'{m:.1f}s ± {s:.1f}s        ', end='')
        print()

    print('-'*70)
    print('Improvement (GCN-MAPPO vs Fixed-time):')
    print(f'{"":20}', end='')
    for d in densities:
        ft = results[d]['Fixed-time']['mean']
        gm = results[d]['GCN-MAPPO']['mean']
        imp = (ft - gm) / ft * 100
        print(f'{imp:+.1f}%                   ', end='')
    print()
    print('='*70)

In [ ]:
import shutil, os

# Only copy individual CSVs to working directory
for f in ['all_points.csv', 'best_points.csv', 'summary.csv']:
    src = f'/content/adaptive_system_results/{f}'
    dst = f'/content/{f}'
    shutil.copy(src, dst)
    print(f'Copied: {f}')

print('\nDone! Now go to Output panel (right side) and download adaptive_system_results.zip')